In [2]:
import heapq
import math
from typing import Dict, List, Optional, Set, Tuple


Graph = Dict[int, List[int]]
Path = List[int]
Length = Dict[Tuple[int,int], float]

def dijkstra(graph: Graph, src: int, dst: int,
             length: Length) -> Optional[Path]:

    dist = {v: math.inf for v in graph}
    dist[src] = 0.0
    prev: Dict[int, Optional[int]] = {v: None for v in graph}
    heap = [(0.0, src)] # Corrected: Added '=' to initialize heap

    while heap:
      d, u = heapq.heappop(heap)
      if d > dist[u]:
        continue
      if u == dst:
        break
      for v in graph[u]:
        e = (min(u,v), max(u,v))
        nd = dist[u] + length[e]
        if nd < dist[v]:
          dist[v] = nd
          prev[v] = u
          heapq.heappush(heap, (nd, v))

    if dist[dst] == math.inf:
      return None

    path, node = [], dst
    while node is not None:
      path.append(node)
      node = prev[node]
    return path[::-1]


def disjoint_paths_pricing(
    graph: Graph,
    edges: List[Tuple[int, int]],
    pairs: List[Tuple[int, int]],
    epsilon: float = 0.1
) -> Tuple[List[Path], Set[Tuple[int,int]], Length]:

  m = len(edges)
  length: Length = {(min(u,v), max(u,v)): 1.0 / m for u, v in edges}

  accepted_paths: List[Path] = []
  accepted_pairs: List[Tuple[int,int]] = []

  for s, t in pairs:
    path = dijkstra(graph, s, t, length)
    if path is None:
      continue

    path_len = sum(
        length[(min(path[i], path[i+1]), max(path[i], path[i+1]))]
        for i in range(len(path) - 1)
    )

    if path_len <= 1.0:
      accepted_paths.append(path)
      accepted_pairs.append((s,t))
      for i in range(len(path) - 1):
        e = (min(path[i], path[i+1]), max(path[i], path[i + 1]))
        length[e] *= (1 + epsilon)

  used_edges: Set[Tuple[int,int]] = set()
  disjoint: List[Path] = []

  for path in accepted_paths:
    path_edges = {
        (min(path[i], path[i+1]), max(path[i], path[i+1]))
        for i in range(len(path) - 1)
    }
    if path_edges.isdisjoint(used_edges):
      disjoint.append(path)
      used_edges |= path_edges

  return accepted_paths, set(map(tuple, disjoint)), length


def lower_bound(disjoint_paths: list) ->int:
  return len(disjoint_paths)



if __name__ == "__main__":
    #  Grafo:
    #  1 — 2 — 5
    #  |   |   |
    #  3 — 4 — 6
    #       \  |
    #        \ |
    #          7
    graph: Graph = {
        1: [2, 3],
        2: [1, 4, 5],
        3: [1, 4],
        4: [2, 3, 6, 7],
        5: [2, 6],
        6: [4, 5, 7],
        7: [4, 6],
    }
    edges = [
        (1,2),(1,3),(2,4),(2,5),(3,4),(4,6),(4,7),(5,6),(6,7)
    ]
    pairs = [(1,6), (3,5), (1,7), (2,6), (3,7)]

    accepted, disjoint, final_lengths = disjoint_paths_pricing(
        graph, edges, pairs, epsilon=0.5
    )

    print("Pares solicitados:", pairs)
    print(f"\nCaminos aceptados (antes de filtrar): {len(accepted)}")
    for p in accepted:
        print(f"  {p}")

    print(f"\nCaminos edge-disjoint finales: {len(disjoint)}")
    for p in disjoint:
        print(f"  {list(p)}")

    print(f"\nLongitudes finales de aristas (top 5 más congestionadas):")
    top = sorted(final_lengths.items(), key=lambda x: -x[1])[:5]
    for e, l in top:
        print(f"  {e}: {l:.4f}")

    print(f"\nLower bound (solución encontrada): {lower_bound(list(disjoint))}")

Pares solicitados: [(1, 6), (3, 5), (1, 7), (2, 6), (3, 7)]

Caminos aceptados (antes de filtrar): 5
  [1, 2, 4, 6]
  [3, 1, 2, 5]
  [1, 3, 4, 7]
  [2, 5, 6]
  [3, 4, 7]

Caminos edge-disjoint finales: 3
  [2, 5, 6]
  [1, 3, 4, 7]
  [1, 2, 4, 6]

Longitudes finales de aristas (top 5 más congestionadas):
  (1, 2): 0.2500
  (1, 3): 0.2500
  (2, 5): 0.2500
  (3, 4): 0.2500
  (4, 7): 0.2500

Lower bound (solución encontrada): 3
